In [1]:
#!pip install scikit-learn

In [11]:
df_train.columns

Index(['Unnamed: 0', 'subject_ids', 'T1_agreement', 'T1', 'T2_agreement', 'T2',
       'T3_agreement', 'T3', 'T4_agreement', 'T4', 'decade', 'filename',
       'T2_norm', 'T1_labels', 'Animal', 'Boy', 'Girl', 'Man', 'Mythical',
       'None', 'Object', 'Other', 'Woman', 'T3_labels', 'Architecture',
       'Household', 'Nature', 'None.1', 'Other.1', 'Reading', 'Toys',
       'Vehicles', 'T4_labels', 'T4_has_label', 'T4_1', 'T4_2', 'T4_3', 'T4_4',
       'T4_5', 'T4_expected', 'original_caption', 'cleaned_caption',
       'structured_caption', 'consensus_caption',
       'structured_consensus_caption'],
      dtype='str')

In [13]:
df_train['T4_expected']

0        2.500000
1        4.000000
2        2.000000
3        2.000000
4        2.333333
           ...   
14093    4.000000
14094    4.000000
14095    4.000000
14096    2.000000
14097    4.000000
Name: T4_expected, Length: 14098, dtype: float64

In [ ]:
import pandas as pd
import numpy as np
import random

from ast import literal_eval

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC


def set_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)

set_seed()

base_path = "./"

df_train = pd.read_csv(f"{base_path}/captions-expanded-train.csv")
df_val = pd.read_csv(f"{base_path}/captions-expanded-val.csv")

TEXT_COLUMNS = [
    ("Original", "original_caption"),
    ("Consensus", "consensus_caption"),
]

T1_COLUMN = "T1_labels"
T3_COLUMN = "T3_labels"

def normalize_text(value):
    if isinstance(value, str):
        return value.strip()
    if pd.isna(value):
        return ""
    return str(value).strip()


def parse_labels(x):
    if pd.isna(x):
        return []
    try:
        out = literal_eval(x)
        if isinstance(out, str):
            return [out]
        return list(out)
    except Exception:
        return []


def compute_metrics(y_true, y_pred):
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "samples_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
    }


def build_data(df, text_col, label_col):
    return (
        df[text_col].map(normalize_text),
        df[label_col].apply(parse_labels)
    )

TFIDF_GRIDS = [(1, 1), (1, 2)]

LR_GRID = [{"C": 2}, {"C": 5}]
SVM_GRID = [{"C": 0.3}, {"C": 0.5}]

def train_model(task_name, text_name, text_col, label_col):

    print(f"\n================ {task_name} | {text_name} =================")

    X_train, y_train_raw = build_data(df_train, text_col, label_col)
    X_val, y_val_raw = build_data(df_val, text_col, label_col)

    mlb = MultiLabelBinarizer()

    y_train = mlb.fit_transform(y_train_raw)
    y_val = mlb.transform(y_val_raw)

    print("Labels:", list(mlb.classes_))
    print("Num labels:", len(mlb.classes_))

    best = {}

    for model_type, grid in [("lr", LR_GRID), ("svm", SVM_GRID)]:

        best_result = None

        for ngram in TFIDF_GRIDS:
            for cfg in grid:

                if model_type == "lr":
                    clf = OneVsRestClassifier(
                        LogisticRegression(
                            C=cfg["C"],
                            class_weight="balanced",
                            max_iter=5000,
                            random_state=42,
                        )
                    )
                else:
                    clf = OneVsRestClassifier(
                        LinearSVC(
                            C=cfg["C"],
                            class_weight="balanced",
                            random_state=42,
                        )
                    )

                model = Pipeline([
                    ("tfidf", TfidfVectorizer(ngram_range=ngram)),
                    ("clf", clf),
                ])

                model.fit(X_train, y_train)

                preds = model.predict(X_val)

                metrics = compute_metrics(y_val, preds)

                print(
                    model_type.upper(),
                    "ngram:", ngram,
                    "cfg:", cfg,
                    metrics
                )

                score = metrics["micro_f1"]

                if best_result is None or score > best_result["score"]:
                    best_result = {
                        "model_type": model_type,
                        "ngram": ngram,
                        "cfg": cfg,
                        "metrics": metrics,
                        "score": score,
                        "model": model,
                    }

        best[model_type] = best_result

    return best

t1_results = {}

for text_name, text_col in TEXT_COLUMNS:
    t1_results[text_name] = train_model(
        task_name="T1",
        text_name=text_name,
        text_col=text_col,
        label_col=T1_COLUMN,
    )

t3_results = {}

for text_name, text_col in TEXT_COLUMNS:
    t3_results[text_name] = train_model(
        task_name="T3",
        text_name=text_name,
        text_col=text_col,
        label_col=T3_COLUMN,
    )

print("\n================ BEST RESULTS =================")

print("\n### T1 RESULTS")
for text_name, results in t1_results.items():
    print(f"\n{text_name}")
    for model_type in ("lr", "svm"):
        r = results[model_type]
        print(model_type.upper(), r["metrics"], r["cfg"])

print("\n### T3 RESULTS")
for text_name, results in t3_results.items():
    print(f"\n{text_name}")
    for model_type in ("lr", "svm"):
        r = results[model_type]
        print(model_type.upper(), r["metrics"], r["cfg"])


================ T1 | Original =================
Labels: ['Animal', 'Boy', 'Girl', 'Man', 'Mythical', 'None', 'Object', 'Other', 'Woman']
Num labels: 9
LR ngram: (1, 1) cfg: {'C': 2} {'micro_f1': 0.8279993693835724, 'macro_f1': 0.57556974532647, 'samples_f1': 0.8097928820151042}
LR ngram: (1, 1) cfg: {'C': 5} {'micro_f1': 0.8275313940549992, 'macro_f1': 0.569550196396536, 'samples_f1': 0.8058666847555737}
LR ngram: (1, 2) cfg: {'C': 2} {'micro_f1': 0.8293861768970517, 'macro_f1': 0.5702378055128058, 'samples_f1': 0.810656627323294}
LR ngram: (1, 2) cfg: {'C': 5} {'micro_f1': 0.8313840155945419, 'macro_f1': 0.5715228333569548, 'samples_f1': 0.8101578257133812}
SVM ngram: (1, 1) cfg: {'C': 0.3} {'micro_f1': 0.829966597741371, 'macro_f1': 0.5692470366921286, 'samples_f1': 0.8085117351784018}
SVM ngram: (1, 1) cfg: {'C': 0.5} {'micro_f1': 0.828125, 'macro_f1': 0.5692051256459494, 'samples_f1': 0.806019536019536}
SVM ngram: (1, 2) cfg: {'C': 0.3} {'micro_f1': 0.8362097036795832, 'macro_f1'

In [ ]:
import pandas as pd
import numpy as np
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVR
from sklearn.metrics import mean_absolute_error

def set_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)

set_seed()

base_path = "./"

df_train = pd.read_csv(f"{base_path}/captions-expanded-train.csv").dropna()
df_val = pd.read_csv(f"{base_path}/captions-expanded-val.csv").dropna()

TEXT_COLUMNS = [
    ("Original", "original_caption"),
    ("Consensus", "consensus_caption"),
]

T4_TARGET = "T4_expected"
DECADE_TARGET = "decade"

AGREEMENT_COLUMN = "T1_agreement"

def normalize_text(x):
    if isinstance(x, str):
        return x.strip()
    if pd.isna(x):
        return ""
    return str(x).strip()


def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {"MAE": mae, "RMSE": rmse}

TFIDF_GRIDS = [(1, 1), (1, 2)]
RIDGE_GRID = [0.5, 1.0, 2.0, 5.0]
SVR_GRID = [0.1, 0.5, 1.0]

def run_regression(task_name, text_name, text_col, target_col, use_sample_weight=False):

    print(f"\n================ {task_name} | {text_name} =================")

    X_train = df_train[text_col].map(normalize_text)
    X_val = df_val[text_col].map(normalize_text)

    y_train = df_train[target_col].values
    y_val = df_val[target_col].values

    sample_weight = None

    if use_sample_weight:
        # higher agreement = higher weight
        sample_weight = df_train[AGREEMENT_COLUMN].map({
            "majority": 1.0,
            "partial_disagreement": 0.75,
            "disagreement": 0.5
        }).fillna(1.0).values

    best = {}

    for model_type, grid in [("ridge", RIDGE_GRID), ("svr", SVR_GRID)]:

        best_result = None

        for ngram in TFIDF_GRIDS:

            for cfg in grid:

                if model_type == "ridge":
                    model = Pipeline([
                        ("tfidf", TfidfVectorizer(ngram_range=ngram)),
                        ("reg", Ridge(alpha=cfg)),
                    ])
                else:
                    model = Pipeline([
                        ("tfidf", TfidfVectorizer(ngram_range=ngram)),
                        ("reg", LinearSVR(C=cfg, max_iter=5000)),
                    ])

                if use_sample_weight:
                    model.fit(X_train, y_train, reg__sample_weight=sample_weight)
                else:
                    model.fit(X_train, y_train)

                preds = model.predict(X_val)

                metrics = compute_metrics(y_val, preds)

                print(
                    model_type.upper(),
                    "ngram:", ngram,
                    "cfg:", cfg,
                    metrics
                )

                score = metrics["MAE"]

                if best_result is None or score < best_result["score"]:
                    best_result = {
                        "model_type": model_type,
                        "ngram": ngram,
                        "cfg": cfg,
                        "metrics": metrics,
                        "score": score,
                        "model": model,
                    }

        best[model_type] = best_result

    return best

results = {}

for text_name, text_col in TEXT_COLUMNS:

    
    results[f"T4_{text_name}"] = run_regression(
        task_name="T4_EXPECTED",
        text_name=text_name,
        text_col=text_col,
        target_col=T4_TARGET,
        use_sample_weight=True
    )


    results[f"DECADE_{text_name}"] = run_regression(
        task_name="DECADE",
        text_name=text_name,
        text_col=text_col,
        target_col=DECADE_TARGET,
        use_sample_weight=False
    )

print("\n================ FINAL RESULTS ================")

for k, v in results.items():
    print(f"\n### {k}")

    for model_type in ("ridge", "svr"):
        r = v[model_type]
        print(model_type.upper(), r["metrics"], r["cfg"])


================ T4_EXPECTED | Original =================
RIDGE ngram: (1, 1) cfg: 0.5 {'MAE': 0.7061352212425107, 'RMSE': np.float64(0.8799632215217671)}
RIDGE ngram: (1, 1) cfg: 1.0 {'MAE': 0.6946651306347857, 'RMSE': np.float64(0.8619131874093606)}
RIDGE ngram: (1, 1) cfg: 2.0 {'MAE': 0.6950864142765286, 'RMSE': np.float64(0.856434954895439)}
RIDGE ngram: (1, 1) cfg: 5.0 {'MAE': 0.7096379387896918, 'RMSE': np.float64(0.8675122240300511)}
RIDGE ngram: (1, 2) cfg: 0.5 {'MAE': 0.7143972819190778, 'RMSE': np.float64(0.8775199176233559)}
RIDGE ngram: (1, 2) cfg: 1.0 {'MAE': 0.707121398835417, 'RMSE': np.float64(0.8672917441935937)}
RIDGE ngram: (1, 2) cfg: 2.0 {'MAE': 0.7099740605260515, 'RMSE': np.float64(0.8678182374371513)}
RIDGE ngram: (1, 2) cfg: 5.0 {'MAE': 0.7302204902116106, 'RMSE': np.float64(0.8880816333030213)}
SVR ngram: (1, 1) cfg: 0.1 {'MAE': 0.7320189970589922, 'RMSE': np.float64(0.8940941641555638)}
SVR ngram: (1, 1) cfg: 0.5 {'MAE': 0.7047197220156876, 'RMSE': np.float6